# RI / SI responses for all tactile sensors

`materials_4x2_blocks_merkel_meissner.pdf` と同じ 4列 x 2行の素材配置で、各素材を RI（上段）と SI（下段）に分けて表示します。

処理順: **センサーゲイン → 指数移動平均 (EMA) → 絶対値微分 |dF/dt| → RI/SI → フィルタゲイン → 描画**

3つのセンサーは色と線種の両方で区別します。補正後の振幅を直接比較できるよう、すべてのパネルで共通のY軸範囲を使用します。

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# Notebookを「いろいろ」またはリポジトリ直下から起動した場合に対応
cwd = Path.cwd().resolve()
if (cwd / 'tactile_data').is_dir():
    PROJECT_ROOT = cwd
elif (cwd.parent / 'tactile_data').is_dir():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError('tactile_data フォルダが見つかりません。研究コード内から実行してください。')

TACTILE_DATA_DIR = PROJECT_ROOT / 'tactile_data'
OUTPUT_DIR = PROJECT_ROOT / 'いろいろ'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MATERIALS = [
    'Al_board', 'buta_omote', 'buta_ura', 'cork',
    'denim', 'rubber_board', 'washi', 'wood_board',
]
MATERIAL_TITLES = {
    'Al_board': 'aluminum bd.',
    'buta_omote': 'outer pigskin',
    'buta_ura': 'back pigskin',
    'cork': 'cork',
    'denim': 'denim',
    'rubber_board': 'rubber bd.',
    'washi': 'Japanese paper',
    'wood_board': 'wood bd.',
}

# 表示する条件。必要に応じてここだけ変更してください。
PLOT_SPEED = 20
SAMPLE_INDEX = 0
START = 3000
END = 8000
DT = 0.1e-3

EMA_TAU_MS = 6.0
RI_TAU_MS = 0.9
SI_TAU_MS = 80.0
DERIVATIVE_WIDTH = 1

# 全素材・全速度・全ファイルから求めたRMSの逆数
SENSOR_GAIN = {
    0: 1.0 / 1.620796,
    1: 1.0 / 2.909883,
    2: 1.0 / 2.279392,
}
FILTER_GAIN = {
    'RI': 1.0 / 9.156278,
    'SI': 1.0 / 553.907547,
}

SENSOR_STYLE = {
    0: {'color': '#D62728', 'linestyle': '-',  'label': 'sensor 1'},
    1: {'color': '#1F77B4', 'linestyle': '--', 'label': 'sensor 2'},
    2: {'color': '#2CA02C', 'linestyle': '-.', 'label': 'sensor 3'},
}

OUT_PDF = OUTPUT_DIR / 'materials_4x2_blocks_RI_SI_all_sensors.pdf'
OUT_PNG = OUTPUT_DIR / 'materials_4x2_blocks_RI_SI_all_sensors.png'

plt.rcParams.update({
    'font.size': 9,
    'axes.linewidth': 0.7,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
})

print('data:', TACTILE_DATA_DIR)
print('output:', OUTPUT_DIR)

data: C:\Users\haru4\研究\研究コード\tactile_data
output: C:\Users\haru4\研究\研究コード\いろいろ


In [2]:
def ema_filter(data, dt=DT, tau_ms=EMA_TAU_MS):
    data = np.asarray(data, dtype=float)
    if data.shape[-1] == 0 or tau_ms <= 0:
        return data.copy()
    alpha = 1.0 - np.exp(-dt / (tau_ms * 1e-3))
    out = np.empty_like(data, dtype=float)
    out[..., 0] = data[..., 0]
    for i in range(1, data.shape[-1]):
        out[..., i] = alpha * data[..., i] + (1.0 - alpha) * out[..., i - 1]
    return out


def abs_derivative(data, dt=DT, width=DERIVATIVE_WIDTH):
    data = np.asarray(data, dtype=float)
    width = int(width)
    if width < 1:
        raise ValueError('width must be >= 1')
    out = np.zeros_like(data, dtype=float)
    out[..., width:] = np.abs(data[..., width:] - data[..., :-width]) / (width * dt)
    return out


def exp_response_filter(data, tau_ms, dt=DT):
    data = np.asarray(data, dtype=float)
    if data.shape[-1] == 0 or tau_ms <= 0:
        return data.copy()
    decay = np.exp(-dt / (tau_ms * 1e-3))
    out = np.zeros_like(data, dtype=float)
    out[..., 0] = data[..., 0]
    for i in range(1, data.shape[-1]):
        out[..., i] = decay * out[..., i - 1] + data[..., i]
    return out


def read_three_sensors(csv_path):
    df = pd.read_csv(
        csv_path, header=None, sep=None, engine='python',
        skiprows=START, nrows=END - START, usecols=[0, 1, 2],
    )
    raw = df.to_numpy(dtype=float).T
    time = np.arange(raw.shape[1], dtype=float) * DT
    return time, raw


def choose_sample(material, speed=PLOT_SPEED, sample_index=SAMPLE_INDEX):
    files = sorted((TACTILE_DATA_DIR / material).glob(f'data_*_x*_z{speed}_p*.csv'))
    if not files:
        raise FileNotFoundError(f'{material}: z={speed} のCSVが見つかりません')
    if not -len(files) <= sample_index < len(files):
        raise IndexError(f'{material}: SAMPLE_INDEX={sample_index}, available={len(files)}')
    return files[sample_index]


def calculate_ri_si(csv_path):
    time, raw = read_three_sensors(csv_path)

    # 1) センサー間の振幅差を補正
    sensor_gain = np.array([SENSOR_GAIN[i] for i in range(raw.shape[0])])[:, None]
    sensor_scaled = raw * sensor_gain

    # 2) RI/SIの前段で指数移動平均を適用
    smoothed = ema_filter(sensor_scaled, dt=DT, tau_ms=EMA_TAU_MS)
    derivative = abs_derivative(smoothed, dt=DT, width=DERIVATIVE_WIDTH)

    # 3) RI/SIを計算後、フィルタ間の振幅差を補正
    ri = exp_response_filter(derivative, tau_ms=RI_TAU_MS, dt=DT) * FILTER_GAIN['RI']
    si = exp_response_filter(derivative, tau_ms=SI_TAU_MS, dt=DT) * FILTER_GAIN['SI']

    return {'time': time, 'RI': ri, 'SI': si, 'file': Path(csv_path)}

In [3]:
# 各素材から同じ速度条件の1ファイルを選び、3センサーすべてを処理
results = {}
selected_rows = []
for material in MATERIALS:
    csv_path = choose_sample(material)
    results[material] = calculate_ri_si(csv_path)
    selected_rows.append({
        'material': material,
        'speed': PLOT_SPEED,
        'file': csv_path.name,
    })

selected_df = pd.DataFrame(selected_rows)
display(selected_df)

# RIとSIを補正後の同一スケールで比較するため、全パネル共通の上限を設定
all_values = np.concatenate([
    result[name].ravel()
    for result in results.values()
    for name in ('RI', 'SI')
])
finite_values = all_values[np.isfinite(all_values)]
COMMON_YMAX = float(np.max(finite_values)) if finite_values.size else 1.0
if not np.isfinite(COMMON_YMAX) or COMMON_YMAX <= 0:
    COMMON_YMAX = 1.0
print(f'common y range: 0 to {COMMON_YMAX:.3g}')

,material,speed,file
0,Al_board,20,data_10_x10000_z20_p1216.csv
1,buta_omote,20,data_10_x10000_z20_p1216.csv
2,buta_ura,20,data_10_x10000_z20_p1216.csv
3,cork,20,data_10_x10000_z20_p1216.csv
4,denim,20,data_10_x10000_z20_p1216.csv
5,rubber_board,20,data_10_x10000_z20_p1216.csv
6,washi,20,data_10_x10000_z20_p1216.csv
7,wood_board,20,data_10_x10000_z20_p1216.csv


common y range: 0 to 3.57


In [4]:
fig = plt.figure(figsize=(11.2, 5.15), dpi=180)
outer = fig.add_gridspec(2, 4, wspace=0.16, hspace=0.48)

for material_index, material in enumerate(MATERIALS):
    block_row, block_col = divmod(material_index, 4)
    inner = outer[block_row, block_col].subgridspec(2, 1, hspace=0.08)
    ax_ri = fig.add_subplot(inner[0, 0])
    ax_si = fig.add_subplot(inner[1, 0], sharex=ax_ri, sharey=ax_ri)
    result = results[material]

    for sensor_id, style in SENSOR_STYLE.items():
        plot_kwargs = dict(
            color=style['color'], linestyle=style['linestyle'],
            linewidth=1.0, alpha=0.95,
        )
        ax_ri.plot(result['time'], result['RI'][sensor_id], **plot_kwargs)
        ax_si.plot(result['time'], result['SI'][sensor_id], **plot_kwargs)

    ax_ri.set_title(MATERIAL_TITLES[material], fontsize=10, pad=3)
    ax_ri.set_ylabel('RI', fontsize=9, labelpad=2)
    ax_si.set_ylabel('SI', fontsize=9, labelpad=2)
    ax_si.set_xlabel('Time [s]', fontsize=9, labelpad=1)
    ax_ri.tick_params(labelbottom=False)

    for ax in (ax_ri, ax_si):
        ax.set_xlim(0.0, (END - START - 1) * DT)
        ax.set_ylim(0.0, COMMON_YMAX * 1.05)
        ax.grid(True, color='0.82', linestyle=':', linewidth=0.55)
        ax.tick_params(axis='both', labelsize=8, length=2, pad=1)

legend_handles = [
    Line2D([0], [0], color=style['color'], linestyle=style['linestyle'],
           linewidth=1.4, label=style['label'])
    for style in SENSOR_STYLE.values()
]
fig.legend(
    handles=legend_handles, loc='upper center', ncol=3, frameon=False,
    bbox_to_anchor=(0.5, 1.015), fontsize=9, handlelength=2.8,
)
fig.subplots_adjust(left=0.055, right=0.995, bottom=0.09, top=0.94)

fig.savefig(OUT_PDF, bbox_inches='tight')
fig.savefig(OUT_PNG, dpi=300, bbox_inches='tight')
print('saved:', OUT_PDF)
print('saved:', OUT_PNG)
plt.show()

saved: C:\Users\haru4\研究\研究コード\いろいろ\materials_4x2_blocks_RI_SI_all_sensors.pdf
saved: C:\Users\haru4\研究\研究コード\いろいろ\materials_4x2_blocks_RI_SI_all_sensors.png


C:\Users\haru4\AppData\Local\Temp\ipykernel_60916\3802454572.py:46: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
